# 21.3 組成 tiny ViT

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/21-transformer/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""追蹤Pre-LN兩次residual，再觀察位置固定與token整體重排的差異。"""

import json

import torch

from miniyolo.vision_data import ColorRectangles
from miniyolo.vision_transformer import TinyViT


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    model = TinyViT().eval()
    image = ColorRectangles(samples=128).images[3:4]
    with torch.no_grad():
        embedded = model.embed_tokens(image)
        block = model.blocks[0]
        attention_update = block.attention(block.norm1(embedded))
        after_attention = embedded + attention_update
        mlp_update = block.mlp(block.norm2(after_attention))
        after_mlp = after_attention + mlp_update
        assert torch.equal(after_mlp, block(embedded))
        features = model.forward_features(image)
        logits = model(image)
        # CLS維持第0位，其餘patch逆序；先不給位置，encoder是排列等變的。
        permutation = torch.cat((torch.tensor([0]), torch.arange(16, 0, -1)))
        no_position = model.embed_tokens(image, use_position=False)
        encoded = model.encode_tokens(no_position)
        reordered_encoded = model.encode_tokens(no_position[:, permutation])
        equivariance_error = float((reordered_encoded - encoded[:, permutation]).abs().max())
        assert torch.allclose(reordered_encoded, encoded[:, permutation], atol=2e-6)
        # 這次只重排patch內容，位置向量仍在原座位；CLS輸出會改變。
        contents_reordered = no_position[:, permutation] + model.pos_embed
        original_with_position = no_position + model.pos_embed
        position_change = float((model.encode_tokens(contents_reordered)[:, 0]
                                 - model.encode_tokens(original_with_position)[:, 0]).abs().max())
        assert position_change > 1e-6
    result = {
        "event": "transformer", "block_count": model.depth,
        "input_tokens_shape": list(embedded.shape), "pre_ln": True,
        "first_attention_update_norm": float(attention_update.norm()),
        "first_mlp_update_norm": float(mlp_update.norm()),
        "manual_residual_matches_block": bool(torch.equal(after_mlp, block(embedded))),
        "cls_feature_shape": list(features["cls"].shape),
        "patch_feature_shape": list(features["patches"].shape), "logit_shape": list(logits.shape),
        "permutation": permutation.tolist(), "without_position_equivariance_max_error": equivariance_error,
        "with_fixed_position_cls_max_change": position_change,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "limitation": "Untrained mechanism check; changed CLS values do not show successful color classification.",
    }
    print(json.dumps(result, ensure_ascii=False, allow_nan=False))
    return result


if __name__ == "__main__":
    main()


{"event": "transformer", "block_count": 2, "input_tokens_shape": [1, 17, 32], "pre_ln": true, "first_attention_update_norm": 0.18654008209705353, "first_mlp_update_norm": 0.2148894965648651, "manual_residual_matches_block": true, "cls_feature_shape": [1, 32], "patch_feature_shape": [1, 16, 32], "logit_shape": [1, 2], "permutation": [0, 16, 15, 14, 13, 12, 11, 10, 9, 8, 7, 6, 5, 4, 3, 2, 1], "without_position_equivariance_max_error": 3.5762786865234375e-07, "with_fixed_position_cls_max_change": 0.03043721616268158, "parameters": 23970, "limitation": "Untrained mechanism check; changed CLS values do not show successful color classification."}
